# 以DataFrame表格形式呈现不同模型给出的正向情感得分


In [ ]:
# 运行准备：按示例代码5.1～5.3和5.43～5.49得到scores_simplenet，并按示例代码5.1～5.2和5.51～5.58得到scores_bert
import os
import csv
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast, BertForSequenceClassification, get_linear_schedule_with_warmup
from tqdm import tqdm

data_file = "../pybook-data/ch5/imdb_labelled.txt"
df = pd.read_csv(data_file, names=["sentence", "label"], sep="\t", quoting=csv.QUOTE_NONE)
sents = df["sentence"].values
y = df["label"].values

out_dir = "output"
train_file = f"{out_dir}/imdb_labelled_train.csv"
test_file = f"{out_dir}/imdb_labelled_test.csv"
if os.path.exists(train_file) and os.path.exists(test_file):
    print(f'Files exist at "{train_file}" and "{test_file}"')
    df_train, df_test = pd.read_csv(train_file), pd.read_csv(test_file)
    sents_train, sents_test = df_train["sentence"].values, df_test["sentence"].values
    y_train, y_test = df_train["label"].values, df_test["label"].values
else:
    os.makedirs(out_dir, exist_ok=True)
    sents_train, sents_test, y_train, y_test = train_test_split(sents, y, test_size=0.2, random_state=1)
    df_train = pd.DataFrame({"sentence": sents_train, "label": y_train})
    df_test = pd.DataFrame({"sentence": sents_test, "label": y_test})
    df_train.to_csv(train_file, index=False)
    df_test.to_csv(test_file, index=False)

fe = CountVectorizer()
fe.fit(sents_train)
vob = fe.get_feature_names_out()

class FeatSet(Dataset):
    def __init__(self, feats, labels):
        self.feats = feats
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        dense = self.feats[idx].toarray().squeeze()
        return torch.tensor(dense, dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)

class SimpleNet(nn.Module):
    def __init__(self, vocab_size, num_class=2):
        super().__init__()
        self.net = nn.Linear(vocab_size, num_class)

    def forward(self, x):
        return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_ds = FeatSet(fe.transform(sents_train), y_train)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
simple_model = SimpleNet(len(vob)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW([p for p in simple_model.parameters() if p.requires_grad], lr=1e-4)
max_epoch = 300
total_steps = len(train_loader) * max_epoch
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
for epoch in range(max_epoch):
    simple_model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = criterion(simple_model(xb), yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

input_texts = ['this is a good movie', 'the movie is horrible', 'a horrible movie', 'not a bad movie']
input_ds = FeatSet(fe.transform(input_texts), [0]*len(input_texts))
input_loader = DataLoader(input_ds, batch_size=16, shuffle=False)
simple_model.eval()
all_scores = []
with torch.no_grad():
    for xb, _ in input_loader:
        output = simple_model(xb.to(device))
        all_scores.append(torch.softmax(output, dim=1).cpu().numpy())
scores_simplenet = np.vstack(all_scores)

tokenizer = BertTokenizerFast.from_pretrained('./bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('./bert-base-uncased', num_labels=2)

class BertDataset(Dataset):
    def __init__(self, sents, labels, tokenizer, max_len=64):
        self.texts = sents
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            add_special_tokens=True,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        item = {key: val.squeeze(0) for key, val in encoded.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = BertDataset(sents_train, y_train, tokenizer)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
max_epoch = 10
total_steps = len(train_loader) * max_epoch
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
for epoch in range(max_epoch):
    model.train()
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        batch = {k: v.to(device) for k, v in batch.items()}
        loss = model(**batch).loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

input_ds = BertDataset(input_texts, [0]*len(input_texts), tokenizer)
input_loader = DataLoader(input_ds, batch_size=16, shuffle=False)
model.eval()
all_scores = []
with torch.no_grad():
    for batch in input_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
        all_scores.append(torch.softmax(outputs.logits, dim=1).cpu().numpy())
scores_bert = np.vstack(all_scores)


In [ ]:
df=pd.DataFrame({'review':input_texts, 'SimpleNet':scores_simplenet[:,1], 'BERT':scores_bert[:,1]})
df